# Naive RAG vs Advanced RAG

This file teaches you two ways to build a RAG pipeline.
By the end, you will know how Naive RAG works, how Advanced RAG improves on it, and when to use each one.

## Setup

In [1]:
!pip install python_dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")

In [3]:
import json
import numpy as np
from openai import OpenAI

client = OpenAI(api_key = api_key)

In [4]:
EMBED_MODEL = "text-embedding-3-small"
MODEL = "gpt-4o-mini"

**What happened:** We imported the libraries and created an OpenAI client. `MODEL` is the LLM we use for generating answers. `EMBED_MODEL` is the model that turns text into number vectors.

In [6]:
def get_embeddings(texts):
    response = client.embeddings.create(
        model=EMBED_MODEL, 
        input=texts
    )

    embeddings = []
    for item in response.data:
        embeddings.append(item.embedding)
    return embeddings

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

**What happened:** We created two helper functions. `get_embeddings` turns a list of text strings into vectors (lists of numbers). `cosine_similarity` measures how close two vectors are. A score of 1.0 means identical, 0.0 means unrelated.

## Knowledge Base

This is our sample data. Each string is one document about Kubernetes.

In [7]:
DOCUMENTS = [
    "HPA automatically adjusts pod replicas based on CPU. Default check interval: 15 seconds. Formula: desiredReplicas = ceil(currentReplicas * currentMetric / desiredMetric).",

    "VPA adjusts CPU and memory requests for pods. Modes: Off, Initial, Auto. Do not use VPA and HPA on the same metrics.",

    "KEDA extends Kubernetes with event-driven autoscaling. Supports 50+ event sources including Kafka, Redis, SQS. Can scale to zero, which HPA cannot do.",
]

**What happened:** We created a small knowledge base with 3 documents about Kubernetes scaling tools.

In [8]:
DOCUMENTS += [
    "Prometheus scrapes /metrics endpoints on pods. PromQL is the query language. Key metric: container_cpu_usage_seconds_total.",

    "Grafana connects to Prometheus for dashboards. Supports alerting via email, Slack, PagerDuty.",

    "Resource requests are guaranteed minimums for scheduling. Limits are hard maximums. Exceeding memory limit causes OOMKill.",
]

**What happened:** We added 3 more documents about monitoring and resource management.

In [9]:
doc_embeddings = np.array(
    get_embeddings(DOCUMENTS), 
    dtype="float32"
)

**What happened:** We turned all 6 documents into vectors. Each document is now a list of numbers. We can compare these numbers to find which documents match a question.

## Naive RAG

**What it does:** Naive RAG is the simplest RAG pipeline. It finds similar documents and puts them into a prompt.

**When to use it:** Prototypes, internal tools, simple question-answering.

**Steps:**
1. Turn the user question into a vector
2. Find the 3 closest documents by cosine similarity
3. Put those documents into a prompt and ask the LLM to answer

In [10]:
def naive_retrieve(query, top_k=3):

    query_emb = get_embeddings([query])[0]
    
    scores = []
    for i in range(len(DOCUMENTS)):
        sim = cosine_similarity(query_emb, 
                                doc_embeddings[i]
                            )
        scores.append((i, sim))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    
    return [(DOCUMENTS[i], s) for i, s in scores[:top_k]]

**What happened:** We built a function that takes a question, turns it into a vector, and finds the top 3 most similar documents.

In [11]:
def naive_rag(query):
    
    results = naive_retrieve(query, top_k=3)

    context_parts = []
    for doc, score in results:
        context_parts.append(doc)
    
    context = "\n\n".join(context_parts)

    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=300, 
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": "Answer based on the context."
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ]
    )

    answer = response.choices[0].message.content
    return answer

**What happened:** The `naive_rag` function retrieves documents, joins them into one string, and sends them to the LLM with the user question.

In [12]:
query = "How do I set up autoscaling with monitoring?"
answer = naive_rag(query)

print(answer)

To set up autoscaling with monitoring in a Kubernetes environment, you can follow these steps:

### Step 1: Set Up Horizontal Pod Autoscaler (HPA)

1. **Install Metrics Server**: Ensure that the Kubernetes Metrics Server is installed in your cluster, as HPA relies on it to gather metrics like CPU usage.
   ```bash
   kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml
   ```

2. **Create HPA Resource**: Define an HPA resource for your application. You can create a YAML file (e.g., `hpa.yaml`) with the following content:
   ```yaml
   apiVersion: autoscaling/v2beta2
   kind: HorizontalPodAutoscaler
   metadata:
     name: my-app-hpa
   spec:
     scaleTargetRef:
       apiVersion: apps/v1
       kind: Deployment
       name: my-app
     minReplicas: 1
     maxReplicas: 10
     metrics:
     - type: Resource
       resource:
         name: cpu
         target:
           type: Utilization
           averageUtilization: 50
   ```

3.

**What happened:** We asked a question and got an answer. Naive RAG works, but it has problems: the question wording might not match the documents, the top 3 results might all cover the same topic, and the answer has no source citations.

## Advanced RAG

**What it does:** Advanced RAG adds several improvement steps to Naive RAG. It rewrites the query, searches with multiple versions, re-ranks results, removes irrelevant parts, and adds source citations.

**When to use it:** Customer-facing products, high-stakes answers, complex questions.

**Steps:**
1. Rewrite the query to be more specific
2. Generate alternative query versions and search with all of them
3. Re-rank results using the LLM
4. Compress each document to keep only relevant sentences
5. Generate an answer with source citations

### Step 1: Query Rewriting

The LLM rewrites the user's question to be more specific and generates alternative versions.

In [15]:
def rewrite_query(query):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=100, 
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system", 
                "content": "Return JSON only."
            },
            {
                "role": "user", 
                "content": (
                "Rewrite this query to be more specific.\n"
                "Also generate 2 alternatives.\n\n"
                f'Query: "{query}"\n\n'
                'Return JSON: {"rewritten": "clearer version", '
                '"alternatives": ["alt1", "alt2"]}')
            }
        ]
    )

    content = response.choices[0].message.content
    parsed = json.loads(content)
    print("Parsed rewritten Queries:\n", parsed)
    
    return parsed

**What happened:** This function asks the LLM to rewrite the query and create 2 alternative versions. Different wordings help find documents that might not match the original query.

### Step 2: Multi-Query Search

We search using all query versions and keep the best score for each document.

In [16]:
def multi_query_search(queries, top_k=5):
    
    all_results = {}
    
    for query in queries:
        query_embedding = get_embeddings([query])[0]
        
        for i in range(len(DOCUMENTS)):
            score = cosine_similarity(
                query_embedding, 
                doc_embeddings[i]
            )

            if i not in all_results:
                all_results[i] = score
            elif score > all_results[i]:
                all_results[i] = score

    ranked = sorted(
        all_results.items(),
        key=lambda x: x[1], reverse=True
    )
    
    results = []
    for i, score in ranked[:top_k]:
        results.append((DOCUMENTS[i], score))
    return results

**What happened:** This function searches with every query version. If two queries both find the same document, it keeps the higher score. This gives broader coverage than a single search.

### Step 3: Re-Ranking

The LLM re-orders the results by relevance to the original question.

In [17]:
def rerank(query, candidates):
    
    doc_lines = []
    
    for i in range(len(candidates)):
        doc = candidates[i][0]
        preview = doc[:100]
        line = f"[DOC {i+1}] {preview}..."
        doc_lines.append(line)
    docs_text = "\n".join(doc_lines)

    
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=100, 
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system", 
                "content": "Return JSON only."
            },
            {
                "role": "user", 
                "content": (
                f"Rank these by relevance to: {query}\n\n"
                f"{docs_text}\n\n"
                'Return JSON: {"rankings": [doc_number, ...]}')
            }
        ]
    )
    
    content = response.choices[0].message.content
    rankings = json.loads(content)["rankings"]

    results = []
    for rank in rankings:
        if 0 < rank <= len(candidates):
            results.append(candidates[rank - 1])
    return results

**What happened:** The LLM looks at all candidate documents and sorts them by how relevant they are to the question. This is more accurate than sorting by vector similarity alone.

### Step 4: Contextual Compression

Remove sentences that are not relevant to the question.

In [18]:
def compress_doc(query, doc):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=100, 
        temperature=0,
        messages=[
            {
                "role": "user", 
                "content": (
                    "Extract ONLY sentences relevant to: "
                    f"{query}\n\nDocument: {doc}\n\n"
                    "Relevant extract (or NONE):")
            }
        ]
    )
    
    extract = response.choices[0].message.content
    extract = extract.strip()
    
    if "NONE" in extract:
        return None
    
    return extract

**What happened:** This function sends each document to the LLM and asks it to keep only the sentences that matter for the question. This reduces noise in the final prompt.

### Step 5: Generate with Citations

Put it all together and generate an answer with source references.

In [ ]:
def advanced_rag(query):
    
    parsed = rewrite_query(query)
    
    all_queries = [parsed["rewritten"]]
    for alternate_query in parsed["alternatives"]:
        all_queries.append(alternate_query)
    print(f"All Queries:\n {all_queries}")

    candidates = multi_query_search(all_queries, top_k=5)
    print("candidates:\n",candidates)

    reranked = rerank(query, candidates)
    for r in reranked:
        print("reranked:\n", r)


    compressed = []
    for i in range(min(4, len(reranked))):
        doc = reranked[i][0]
        result = compress_doc(query, doc)
        print("compressed result:\n", result)
        if result:
            compressed.append(result)
    return compressed


Parsed rewritten Queries:
 {'rewritten': 'What are the steps to configure autoscaling for my application while ensuring effective monitoring of its performance?', 'alternatives': ['How can I implement autoscaling in my cloud environment and set up monitoring tools to track resource usage?', 'What specific configurations are needed to enable autoscaling and integrate monitoring solutions for my server infrastructure?']}
All Queries:
 ['What are the steps to configure autoscaling for my application while ensuring effective monitoring of its performance?', 'How can I implement autoscaling in my cloud environment and set up monitoring tools to track resource usage?', 'What specific configurations are needed to enable autoscaling and integrate monitoring solutions for my server infrastructure?']
candidates:
 [('KEDA extends Kubernetes with event-driven autoscaling. Supports 50+ event sources including Kafka, Redis, SQS. Can scale to zero, which HPA cannot do.', np.float64(0.4131132857726867

**What happened:** This function runs all the steps: rewrite, search, re-rank, compress.

In [32]:
query = "How do I set up autoscaling with monitoring?"
compressed = advanced_rag(query)

source_lines = []
for i in range(len(compressed)):
    line = f"[Source {i+1}]: {compressed[i]}"
    source_lines.append(line)
numbered = "\n".join(source_lines)

print(f"\nCompressed sources:")
print(numbered)

Parsed rewritten Queries:
 {'rewritten': 'What are the steps to configure autoscaling for my application while ensuring effective monitoring of its performance?', 'alternatives': ['How can I implement autoscaling in my cloud environment and set up monitoring tools to track resource usage?', 'What specific configurations are needed to enable autoscaling and integrate monitoring solutions for my server infrastructure?']}
All Queries:
 ['What are the steps to configure autoscaling for my application while ensuring effective monitoring of its performance?', 'How can I implement autoscaling in my cloud environment and set up monitoring tools to track resource usage?', 'What specific configurations are needed to enable autoscaling and integrate monitoring solutions for my server infrastructure?']
candidates:
 [('KEDA extends Kubernetes with event-driven autoscaling. Supports 50+ event sources including Kafka, Redis, SQS. Can scale to zero, which HPA cannot do.', np.float64(0.4131132857726867

In [31]:
response = client.chat.completions.create(
    model=MODEL, 
    max_tokens=400, 
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "Answer using ONLY the sources. Cite with [Source N]."
        },
        {
            "role": "user",
            "content": f"Sources:\n{numbered}\n\nQuestion: {query}"
        }
    ]
)
answer = response.choices[0].message.content
print(f"Answer:\n{answer}")

Answer:
To set up autoscaling with monitoring, follow these steps:

1. **Choose Your Cloud Provider**: Select a cloud provider that supports autoscaling, such as AWS, Google Cloud, or Azure.

2. **Set Up Monitoring Tools**: Implement monitoring tools that can track the performance metrics of your application. This could include CPU usage, memory usage, or request counts. Most cloud providers offer built-in monitoring services (e.g., AWS CloudWatch, Google Cloud Monitoring).

3. **Define Autoscaling Policies**: Create policies that determine when to scale in (reduce instances) and scale out (add instances). These policies should be based on the metrics collected by your monitoring tools. For example, you might set a policy to add instances when CPU usage exceeds 70% for a sustained period.

4. **Configure Autoscaling Groups**: Set up autoscaling groups in your cloud provider's management console. Specify the minimum and maximum number of instances, as well as the scaling policies you de

**What happened:** The Advanced RAG answer includes source citations. Each claim points back to a specific source document so the user can verify the information.

## Comparison: Naive vs Advanced RAG

| Feature | Naive RAG | Advanced RAG |
|---|---|---|
| API calls per query | 2 | ~10 |
| Latency | ~1 second | ~3-5 seconds |
| Cost per query | ~$0.001 | ~$0.005-0.01 |
| Source citations | No | Yes |
| Handles vague queries | Poorly | Well |
| Best for | Prototyping, internal tools | Customer-facing, high-stakes |

## Summary

- **Naive RAG** has 3 steps: embed the query, find similar documents, generate an answer
- **Advanced RAG** adds: query rewriting, multi-query search, re-ranking, compression, and citations
- Query rewriting helps when the user's wording does not match the documents
- Multi-query search finds more documents by searching with different versions of the question
- Re-ranking uses the LLM to sort results by actual relevance, not just vector similarity
- Contextual compression removes irrelevant sentences before generating the answer
- Advanced RAG costs more and is slower, but produces better answers with source attribution
- Start with Naive RAG for a baseline, then add Advanced RAG improvements where needed